# Week 9 Task 1 - Model Evaluation & Hyperparameter Tuning
Random Forest + K-fold CV + GridSearchCV + learning curves + ROC analysis

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV, learning_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve


In [ ]:
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print('Dataset:', X.shape)
print('Missing values:', int(X.isna().sum().sum()))


In [ ]:
baseline = RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1)
start = time.perf_counter(); baseline.fit(X_train, y_train); baseline_time = time.perf_counter()-start
baseline_pred = baseline.predict(X_test); baseline_prob = baseline.predict_proba(X_test)[:,1]
baseline_auc = roc_auc_score(y_test, baseline_prob)
baseline_scores = {
    'Accuracy': accuracy_score(y_test, baseline_pred),
    'Precision': precision_score(y_test, baseline_pred),
    'Recall': recall_score(y_test, baseline_pred),
    'F1': f1_score(y_test, baseline_pred),
    'ROC-AUC': baseline_auc,
    'Training Time (s)': baseline_time
}
print(baseline_scores)
cv_baseline = cross_val_score(baseline, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
print('Baseline CV ROC-AUC:', cv_baseline.mean(), '+/-', cv_baseline.std())


In [ ]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2']
}
search = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid=param_grid,
    scoring='roc_auc', cv=cv, n_jobs=-1, return_train_score=True
)
start = time.perf_counter(); search.fit(X_train, y_train); search_time = time.perf_counter()-start
print('Best parameters:', search.best_params_)
print('Best CV ROC-AUC:', search.best_score_)
print('Grid search time:', search_time)


In [ ]:
best = search.best_estimator_
pred = best.predict(X_test); prob = best.predict_proba(X_test)[:,1]
optimized_scores = {
    'Accuracy': accuracy_score(y_test, pred),
    'Precision': precision_score(y_test, pred),
    'Recall': recall_score(y_test, pred),
    'F1': f1_score(y_test, pred),
    'ROC-AUC': roc_auc_score(y_test, prob),
    'Training Time (s)': search_time
}
comparison = pd.DataFrame({'Baseline': baseline_scores, 'Optimized': optimized_scores})
display(comparison.round(4))


In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    best, X_train, y_train, cv=cv, scoring='roc_auc',
    train_sizes=np.linspace(0.1, 1.0, 5), n_jobs=-1
)
train_mean=train_scores.mean(axis=1); val_mean=val_scores.mean(axis=1)
plt.figure(figsize=(8,5)); plt.plot(train_sizes, train_mean, marker='o', label='Training ROC-AUC'); plt.plot(train_sizes, val_mean, marker='o', label='Validation ROC-AUC')
plt.xlabel('Training examples'); plt.ylabel('ROC-AUC'); plt.title('Learning Curve'); plt.legend(); plt.grid(alpha=0.25); plt.tight_layout(); plt.savefig('../plots/learning_curves.png', dpi=150); plt.show()


In [ ]:
fpr, tpr, _ = roc_curve(y_test, prob)
plt.figure(figsize=(7,5)); plt.plot(fpr, tpr, label=f'Optimized ROC-AUC = {roc_auc_score(y_test, prob):.3f}'); plt.plot([0,1],[0,1],'--', label='Chance')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.title('ROC Curve'); plt.legend(); plt.grid(alpha=0.25); plt.tight_layout(); plt.savefig('../plots/roc_curve.png', dpi=150); plt.show()


## Interpretation
Use the comparison table to discuss performance gains. Use the learning curve to determine whether the model shows high bias or high variance. The ROC-AUC and held-out test metrics should guide final model selection.